# Day 2 · Hands-on 6 — Differential expression and the false discovery rate
**About 35 min · work in pairs** (one drives, one reads, predicts and questions — swap halfway)


An RNA-seq experiment tests **thousands of genes at once**. Yesterday you saw that 5% of tests with no real effect come out "significant". Here we see what that means for 20,000 genes, and how the **false discovery rate (FDR)** keeps the list of hits trustworthy.



> **Two AI roles — keep them apart**
> - 🧮 **Gemini in Colab = calculator.** Ask it to write, change, fix or explain *code*, including new simulations.
> - 🎓 **Claude / ChatGPT (other tab) = tutor.** Use it to explain ideas, quiz you and critique your own work. Ask for its reasoning, and check one claim yourself in the notebook.


**How this notebook works:** run cells from top to bottom. Each code cell has one line marked `# 🔧 CHANGE THIS` — the only thing you need to edit. Lines starting with 📋 go in the shared sheet. Lines starting with 🎯 appear only for the simulated dataset, where we know the true answer.

In [ ]:
# @title ▶️ Setup: run this cell first (about 1 minute) { display-mode: "form" }
%pip install -q pydeseq2
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from statsmodels.stats.multitest import multipletests
from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats

# ---- RNA-seq datasets: a counts table (genes × samples) + a sample table ----
AIRWAY = "https://raw.githubusercontent.com/bioboot/bggn213_W19/master/class-material/airway_scaledcounts.csv"
AIRWAY_META = "https://raw.githubusercontent.com/bioboot/bggn213_W19/master/class-material/airway_metadata.csv"

def _airway():
    counts = pd.read_csv(AIRWAY, index_col=0)
    meta = pd.read_csv(AIRWAY_META, index_col=0)
    return counts, meta, None

def _sim_rnaseq(seed=7, n_genes=8000, n_donors=4, frac_de=0.10):
    rng = np.random.default_rng(seed)
    base = rng.lognormal(5, 1.8, n_genes)                       # typical expression of each gene
    disp = 0.03 + 1.5 / np.sqrt(base)                           # biological overdispersion
    is_de = rng.random(n_genes) < frac_de
    lfc = np.where(is_de, rng.choice([-1, 1], n_genes) * rng.uniform(0.5, 2.5, n_genes), 0)
    donor = rng.normal(0, 0.5, (n_donors, n_genes))             # donor-to-donor differences
    cols, rows = {}, []
    for dnr in range(n_donors):
        for cond, k in [("control", 0), ("treated", 1)]:
            mu = base * np.exp(donor[dnr]) * 2 ** (lfc * k) * rng.uniform(0.7, 1.3)
            name = f"donor{dnr + 1}_{cond}"
            cols[name] = rng.negative_binomial(1 / disp, 1 / (1 + disp * mu))
            rows.append({"sample": name, "condition": cond, "donor": f"donor{dnr + 1}"})
    genes = [f"gene{i:05d}" for i in range(n_genes)]
    return (pd.DataFrame(cols, index=genes), pd.DataFrame(rows).set_index("sample"),
            pd.Series(is_de, index=genes))

def _my_rnaseq():
    # Upload my_counts.csv (genes as rows, samples as columns) and my_metadata.csv (one row per sample)
    return pd.read_csv("my_counts.csv", index_col=0), pd.read_csv("my_metadata.csv", index_col=0), None

RNASEQ = {
    "airway": dict(load=_airway, about="Airway smooth muscle cells ± dexamethasone (GSE52778): 4 cell lines × 2",
                   condition="dex", reference="control", treated="treated", block="celltype"),
    "simulated": dict(load=_sim_rnaseq, about="Simulated: 8,000 genes, 4 donors × 2 conditions, 10% truly DE",
                      condition="condition", reference="control", treated="treated", block="donor"),
    "my_data": dict(load=_my_rnaseq, about="Your own counts: edit these roles",
                    condition="my_condition", reference="my_control_level", treated="my_treated_level", block=None),
}

def load_rnaseq(name):
    S = dict(RNASEQ[name])
    counts, meta, truth = S.pop("load")()
    counts = counts.round().astype(int)
    meta = meta.loc[counts.columns]
    S["truth"] = truth
    print(f"✅ {S['about']}")
    print(f"   {counts.shape[0]} genes × {counts.shape[1]} samples;  condition column: '{S['condition']}';  block: {S['block']}")
    return counts, meta, S

print("✅ Setup complete")

## A. 🧩 Puzzle (run before the lecture)
10,000 genes, 3 control vs 3 treated samples, and **no gene truly changes**. We t-test every gene.

✍️ **Predict first:** how many genes will have p < 0.05? Use your **pair number** as the seed.

In [ ]:
# 🔧 CHANGE THIS: your pair number
seed = 1

rng = np.random.default_rng(seed)
expr = rng.normal(8, 1, size=(10000, 6))                     # log-expression, no real differences
p = stats.ttest_ind(expr[:, :3], expr[:, 3:], axis=1).pvalue
hits = (p < 0.05).sum()
print(f"Genes with p < 0.05: {hits} of 10,000")
print(f"Smallest p-value: {p.min():.2g}")
print("📋 For the shared sheet:")
print(f"   seed = {seed}   |   'significant' genes = {hits}")

## B. Three ways to choose your hits, with the truth known
### 🧭 Concept
Now some genes *are* truly differentially expressed. Compare three rules:
- **Raw p < 0.05**: controls the error rate per gene, so many false hits.
- **Bonferroni**: p < 0.05 / number of tests. Controls the chance of even **one** false hit; very strict.
- **Benjamini–Hochberg (BH)**: controls the **false discovery rate**, the expected *fraction* of false hits among the genes you call.

Try different fractions of truly changed genes: 0, 0.01, 0.1, 0.3.

In [ ]:
# 🔧 CHANGE THIS: fraction of genes that truly change
fraction_de = 0.10

rng = np.random.default_rng()
n_genes, n = 10000, 5
truly_de = rng.random(n_genes) < fraction_de
expr = rng.normal(8, 1, size=(n_genes, 2 * n))
expr[truly_de, n:] += rng.choice([-3, 3], truly_de.sum())[:, None]     # real changes in treated samples
p = stats.ttest_ind(expr[:, :n], expr[:, n:], axis=1).pvalue

rows = []
for name, called in [("raw p < 0.05", p < 0.05),
                     ("Bonferroni", multipletests(p, 0.05, "bonferroni")[0]),
                     ("BH (FDR 5%)", multipletests(p, 0.05, "fdr_bh")[0])]:
    false = (called & ~truly_de).sum()
    rows.append({"rule": name, "genes called": called.sum(), "false hits": false,
                 "false discovery proportion": f"{false / max(called.sum(), 1):.1%}",
                 "true genes found": f"{(called & truly_de).sum()} / {truly_de.sum()}"})
display(pd.DataFrame(rows).set_index("rule"))

### ✅ Concept check
1. With `fraction_de = 0`, how many genes does BH call? How does that compare with raw p < 0.05?
2. Why does BH find more true genes than Bonferroni while keeping the false proportion near 5%?

<details><summary>Show answers</summary>

1. Usually none (or very few): with no real signal, BH protects you as Bonferroni does.
2. Bonferroni protects against *any* false hit, which is very demanding with 10,000 tests. BH accepts that ~5% of the *list* may be false and in exchange keeps far more power. For discovery-oriented screens that is usually the right trade.
</details>

## C. Real differential expression with DESeq2
### 🧭 Concept
DESeq2 fits a **negative binomial model for every gene** (Hands-on 5), tests the treatment effect, and reports BH-adjusted p-values (`padj`). Samples often come in **blocks**, such as donors or cell lines. Including the block in the model is the RNA-seq version of the paired t-test from Day 1.

Choose the dataset first.

In [ ]:
# 🔧 CHANGE THIS: which RNA-seq dataset to use
rnaseq_dataset = "airway"  # @param ["airway", "simulated", "my_data"]

counts, meta, S = load_rnaseq(rnaseq_dataset)
meta

In [ ]:
# 🔧 CHANGE THIS: "paired" (block + condition) or "unpaired" (condition only)
design = "paired"

cond, block = S["condition"], S["block"]
formula = f"~{block} + {cond}" if (design == "paired" and block) else f"~{cond}"
X = counts.T.loc[:, counts.sum(axis=1) >= 10]                 # samples × genes; drop nearly empty genes
print(f"Running DESeq2 with design {formula} on {X.shape[1]} genes (about 1 minute)...")
dds = DeseqDataSet(counts=X, metadata=meta, design=formula, quiet=True)
dds.deseq2()
ds = DeseqStats(dds, contrast=[cond, S["treated"], S["reference"]], quiet=True)
ds.summary()
res = ds.results_df.dropna(subset=["pvalue"])
RESULTS = globals().setdefault("RESULTS", {})
RESULTS[design] = res

n_tested = len(res)
print(f"\nGenes tested:                        {n_tested}")
print(f"Raw p < 0.05:                        {(res.pvalue < 0.05).sum()}   (expected by chance alone if nothing changed: ~{0.05 * n_tested:.0f})")
print(f"BH-adjusted p (padj) < 0.05:         {(res.padj < 0.05).sum()}")
print(f"Bonferroni (p < 0.05 / {n_tested}):  {(res.pvalue < 0.05 / n_tested).sum()}")
print("📋 For the shared sheet:")
print(f"   design = {design}   |   genes with padj < 0.05 = {(res.padj < 0.05).sum()}")
if S["truth"] is not None:
    called = res.index[res.padj < 0.05]
    print(f"🎯 Of {len(called)} genes called, {(~S['truth'][called]).sum()} are truly unchanged "
          f"({(~S['truth'][called]).mean():.1%} false discoveries); {S['truth'][called].sum()} of {S['truth'].sum()} true genes found")

sig = res.padj < 0.05
plt.figure(figsize=(6, 4.5))
plt.scatter(res.log2FoldChange, -np.log10(res.pvalue), c=np.where(sig, "crimson", "lightgray"), s=4)
plt.xlabel("log2 fold change (treated vs control)"); plt.ylabel("−log10 p-value")
plt.title(f"Volcano plot ({design}): red = padj < 0.05"); plt.show()

Run the cell with both designs. If both are run, this compares them:

In [ ]:
# 🔧 CHANGE THIS: FDR threshold
fdr = 0.05

for k, r in globals().get("RESULTS", {}).items():
    print(f"{k:9s}: {(r.padj < fdr).sum():6d} genes with padj < {fdr}")

## D. ⭐ If you have time: what the p-value histogram tells you
### 🧭 Concept
Recall Day 1: null p-values are **flat**. Real genome-wide p-values look like a flat floor (unchanged genes) plus a spike near 0 (changed genes). The height of the floor estimates how many genes are unchanged.

In [ ]:
# 🔧 CHANGE THIS: which run to look at — "paired" or "unpaired"
which = "paired"

if which not in globals().get("RESULTS", {}):
    raise SystemExit(f"Run section C with design = '{which}' first.")
r = RESULTS[which]
plt.figure(figsize=(6, 3.5))
plt.hist(r.pvalue, bins=40, range=(0, 1), color="gray", edgecolor="white")
floor = len(r) / 40 * 2 * np.mean(r.pvalue > 0.5)
plt.axhline(floor, color="red", ls="--", label="estimated floor of unchanged genes")
plt.xlabel("p-value"); plt.ylabel("Number of genes"); plt.legend(); plt.show()
print(f"Estimated fraction of unchanged genes: {2 * np.mean(r.pvalue > 0.5):.0%}")

## 🤖 Go deeper with AI
Three ways to keep learning about differential expression and FDR: now, if you finish early, or after the workshop.

**🎓 Explain and quiz me** (Claude / ChatGPT):
> *Explain the difference between a p-value, a Bonferroni-adjusted p-value and a Benjamini–Hochberg adjusted p-value using a 10-gene example. Then quiz me on what padj = 0.03 does and doesn't mean.*

**🧪 See it in a simulation** (🧮 Gemini in Colab, in a new cell at the end of this notebook):
> *Simulate 10,000 genes of which 10% are truly changed. Apply BH at 5% and compute the actual share of false discoveries in the list. Repeat 100 times and plot the distribution.*

**🔬 Apply it to my work** (Claude / ChatGPT; replace the brackets with your own details):
> *Here is my RNA-seq design: [samples, conditions, batches, donors]. Write the DESeq2 design formula, and tell me what to check before I trust the gene list.*

✔ **Check one claim yourself:** The false discovery share should average at or below 5%, but single runs vary around it.

## 💬 Discuss with your partner
- Why did the paired design find more genes? Link it to the paired t-test card from Day 1.
- You follow up the top 20 genes by qPCR. Roughly how many would you expect to fail to validate at FDR 5%? Why might it be fewer?
- Your colleague tested 12 cytokines and reports raw p-values. Should they use Bonferroni or BH? Does the answer depend on the goal?